# Implicit Decision Gate

## Motivation

Long-running AI work can quietly make important choices that the original request never made. A request to add an export feature might not say who may export, how long exported files should be kept, or whether each export must be recorded. The code must still choose a behavior, and that choice can be hard to notice inside a large change.

The larger idea behind this project is one shared gate for these missing decisions. Separate checks for important parts of a system report simple facts about what the agent actually changed. A database check can report what happens to existing data, a permission check can report who gained access, a storage check can report how long data is kept, and an API check can report behavior visible to other software. If a reported fact matters and the request contains no approved answer for it, the gate saves the work and asks a person.

This scales by building each kind of check once and reusing it across many jobs. A permission check does not need to understand every feature, it only reports permission changes. A storage check only reports changes to how long data is kept. The shared gate handles saving, asking, resuming, and checking the next result for all of them. It doesn't promise to find every possible hidden choice. It covers important parts of a system where effects can be observed reliably.

This notebook walks through the default PostgreSQL example. PostgreSQL reports whether the proposed change preserves existing links or makes them expire. The request doesn't choose between those outcomes, so the gate pauses, records the owner's answer, starts a fresh Codex process, and checks the result again. The repository also implements a workspace export authorization scenario that reports two independent decisions in one run. Start it with `uv run idg start --scenario workspace-export-authorization`.

## Premise

Implicit Decision Gate is a deliberately small, fictional contract-completion stage inside the trust architecture described in 1Password's [Verified Loops](https://1password.com/blog/verified-loops-building-ai-agent-trust). That architecture makes the human-owned job definition the verification boundary and leaves humans the consequential judgments that cannot be verified mechanically. This notebook makes one such boundary executable: what happens when a system-observed effect reveals that the request never made a required choice?

Imagine a service behind 1Password item-sharing links. A customer can share a 1Password item, such as a Login item containing a password, by link. A brief asks a coding agent to make newly created links expire after 30 days. The service stores link records in a PostgreSQL table named `public.share_links`.

The brief doesn't say what should happen to links customers already created. A valid PostgreSQL migration must nevertheless choose between two very different outcomes:

| Decision | Existing links | New links |
| --- | --- | --- |
| `PRESERVE_EXISTING` | Keep their current non-expiring behavior | Expire after 30 days |
| `EXPIRE_EXISTING` | Expire 30 days after migration | Expire after 30 days |

Either policy could be legitimate. Expiring old links can break customer workflows, but preserving them can retain access longer than the new policy intends. The agent shouldn't silently invent the answer.

This scenario is fictional. It makes no claim about 1Password's production services, database schema, or implementation of item sharing. In `public.share_links`, `public` is only the standard PostgreSQL schema namespace. It doesn't mean the table or the links are publicly accessible.

The authoritative brief inspected below is written as an ordinary engineering request and is passed to Codex verbatim. It contains no disclaimer telling the agent that it is participating in a demo so as not to skew any agentic behavior.

## Notebook structure

The notebook uses the same public `idg` commands as the terminal demo and then shows the saved information from each step. The plain `start` command selects the default share-link scenario, `answer` saves the person's choice, and `resume` creates and checks the second attempt. Run those cells once from top to bottom. To repeat the walkthrough, rerun from `start` and continue with the new run identifier.

This checked-in copy includes outputs from the original database-only live execution. Its displayed record and commands use the earlier singular schema; rerunning the current source cells uses `outcomes`, `decisions`, and `decision_requests` and creates new identifiers, timestamps, SQL, and model answers.

## System context

The prototype completes one missing part of a human-owned contract and returns the result to the wider verified loop. Authenticated identities, controlled tools, attributable evidence, and permission enforcement remain responsibilities of the surrounding Verified Loops architecture.

![System context: the human brief and pinned Git inputs flow through the gate, Codex, PostgreSQL, evidence review, durable storage, and back to the verified loop.](assets/diagrams/system_context.png)

[Review the Mermaid source.](assets/diagrams/system_context.mmd)

## 1. Establish the notebook boundary

This cell locates the repository and defines small display helpers. All state-changing work still goes through `uv run idg`. The helpers only execute commands and read the files that the application persists.

In [1]:
from __future__ import annotations

import copy
import difflib
import hashlib
import json
import os
import shlex
import shutil
import stat
import subprocess
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "examples/share-link-expiration/brief.md"
        ).is_file():
            return candidate.resolve()
    raise RuntimeError("Open this notebook from inside the implicit-decision-gate repository")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
COMMAND_ENV = os.environ.copy()
COMMAND_ENV.pop("VIRTUAL_ENV", None)


def run_command(
    arguments: list[str],
    *,
    show_output: bool = True,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    completed = subprocess.run(
        arguments,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=False,
        env=COMMAND_ENV,
    )
    if show_output:
        print(f"$ {shlex.join(arguments)}")
        if completed.stdout:
            print(completed.stdout.rstrip())
        if completed.stderr:
            print(completed.stderr.rstrip())
    if check and completed.returncode != 0:
        raise RuntimeError(
            f"Command exited with status {completed.returncode}: {shlex.join(arguments)}"
        )
    return completed


def run_idg(*arguments: str) -> dict[str, Any]:
    completed = run_command(["uv", "run", "idg", *arguments], check=False)
    try:
        payload = json.loads(completed.stdout)
    except json.JSONDecodeError as error:
        raise RuntimeError("idg did not return its expected JSON summary") from error
    if completed.returncode != 0:
        print(
            f"idg exited with status {completed.returncode}; "
            "its persisted summary remains inspectable"
        )
    return payload


def load_run(run_id: str) -> dict[str, Any]:
    path = REPO_ROOT / ".idg" / "runs" / run_id / "run.json"
    return json.loads(path.read_text(encoding="utf-8"))


print(f"Repository: {REPO_ROOT}")

Repository: /home/user/projects/implicit-decision-gate


## 2. Check the execution environment

The live path needs Git and `uv`, an installed and authenticated Codex CLI, and Docker with Compose. Compose supplies the disposable PostgreSQL 17 verifier.

The application pins every model process to `gpt-5.6-terra` with `xhigh` reasoning. The version identifies the local harness, while the model and reasoning effort identify the requested inference configuration. Each invocation is recorded in `run.json`.

In [2]:
from implicit_decision_gate.codex_client import CODEX_MODEL, CODEX_REASONING_EFFORT

required_tools = ["git", "uv", "codex", "docker"]

tool_paths = {name: shutil.which(name) for name in required_tools}
print(json.dumps(tool_paths, indent=2))
missing_tools = [name for name, path in tool_paths.items() if path is None]
if missing_tools:
    raise RuntimeError(f"Missing required tools: {', '.join(missing_tools)}")

run_command(["git", "rev-parse", "HEAD"])
run_command(["uv", "--version"])
run_command(["codex", "--version"])
print("Application-pinned model configuration:")
print(
    json.dumps(
        {"model": CODEX_MODEL, "reasoning_effort": CODEX_REASONING_EFFORT},
        indent=2,
    )
)
_ = run_command(["docker", "compose", "version"])

{
  "git": "/usr/bin/git",
  "uv": "/home/user/.local/bin/uv",
  "codex": "/home/user/.local/bin/codex",
  "docker": "/usr/bin/docker"
}
$ git rev-parse HEAD
43c97f5306c14d26043b8dfb25dc79f866167876
$ uv --version
uv 0.11.25 (x86_64-unknown-linux-gnu)
$ codex --version
codex-cli 0.149.0
Application-pinned model configuration:
{
  "model": "gpt-5.6-terra",
  "reasoning_effort": "xhigh"
}
$ docker compose version
Docker Compose version v5.3.0


## 3. Start the disposable verifier

Compose starts the PostgreSQL 17 verifier. PostgreSQL is part of the trust argument: it executes the real DDL and exposes default, backfill, nullability, and rollback behavior that SQL text inspection alone can't establish.

In [3]:
_ = run_command(["docker", "compose", "up", "-d", "--wait"])

$ docker compose up -d --wait
 Container implicit-decision-gate-postgres-1 Running 
 Container implicit-decision-gate-postgres-1 Waiting 
 Container implicit-decision-gate-postgres-1 Healthy


## 4. Inspect the authoritative inputs

The brief and schema below are read from the current commit. The seeded `existing-fixture` row makes the missing rollout policy observable. At this point the brief is the exact source text and there is no precomputed semantic representation.

In [4]:
HEAD_BEFORE_START = run_command(["git", "rev-parse", "HEAD"], show_output=False).stdout.strip()
BRIEF_PATH = "examples/share-link-expiration/brief.md"
SCHEMA_PATH = "examples/share-link-expiration/schema.sql"
brief_at_head = run_command(
    ["git", "show", f"{HEAD_BEFORE_START}:{BRIEF_PATH}"], show_output=False
).stdout
schema_at_head = run_command(
    ["git", "show", f"{HEAD_BEFORE_START}:{SCHEMA_PATH}"], show_output=False
).stdout

print(f"Pinned candidate commit: {HEAD_BEFORE_START}")
print("\nAuthoritative brief:\n")
print(brief_at_head)
print("Baseline schema and fixture:\n")
print(schema_at_head)

Pinned candidate commit: 43c97f5306c14d26043b8dfb25dc79f866167876

Authoritative brief:

Add 30-day expiration support to item-sharing links.

Store expiration in `public.share_links.expires_at` as a nullable timestamp with time
zone. New item-sharing links must expire 30 days after creation.

Baseline schema and fixture:

CREATE TABLE public.share_links (
    id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    token text NOT NULL UNIQUE,
    created_at timestamp with time zone NOT NULL DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO public.share_links (token) VALUES ('existing-fixture');



### Brief, context, and rendered prompt

The brief is an engineering ticket. The rendered prompt is not a second ticket and is not an engineer's manual reinterpretation of it. The gate application deterministically combines independently owned artifacts into the complete input for one ephemeral Codex process.

| Artifact | Owner | Role |
| --- | --- | --- |
| Authoritative brief | Human job owner | Product intent and verification boundary |
| Baseline schema | Service repository, pinned by Git | Technical context and observable fixture |
| Prompt envelope | Gate application | Execution isolation and structured-output instructions |
| Rendered prompt | Gate prompt renderer | Materialized envelope, verbatim brief, schema, and any approved amendment |
| Owner amendment | Human job owner | The smallest missing judgment added to attempt two |

The brief appears separately and inside each applicable prompt for two reasons. First, every Codex process is ephemeral and must receive its complete input. Second, `run.json` retains both the source contract and the exact materialized prompt so an auditor can verify that the application didn't silently translate or replace the owner's words.

The prompt envelope contains execution and output constraints. Attempt two additionally contains the typed owner decision, its required behavior, and the PostgreSQL acceptance criteria derived from that explicit amendment.

## Lifecycle at a glance

This swimlane shows the reference path that the remaining cells unpack. Attempt two is a new process, separate from attempt one.

![Lifecycle: the gate saves attempt one, checks it in PostgreSQL, pauses for an unsupported decision, records the owner's choice, and verifies a fresh second attempt.](assets/diagrams/lifecycle.png)

[Review the Mermaid source.](assets/diagrams/lifecycle.mmd)

## 5. Start one durable run

`start` pins the current commit, creates a clean detached worktree, invokes a fresh Codex process for SQL, probes that SQL in PostgreSQL, and invokes a separate fresh Codex process for the evidence question. It returns only after those stages are persisted. The following cells inspect the captured start snapshot in their causal order. Because the model calls are live, an unexpected terminal state is displayed and stops this walkthrough instead of being forced into the expected story. Rerun this cell to create another independent run.

In [5]:
start_summary = run_idg("start")
RUN_ID = str(start_summary["run_id"])
RUN_DIR = REPO_ROOT / ".idg" / "runs" / RUN_ID
START_SNAPSHOT = copy.deepcopy(load_run(RUN_ID))
DECISION_ID = "existing_item_sharing_link_rollout"
START_DECISION = START_SNAPSHOT["decisions"][0]
assert START_DECISION["decision_id"] == DECISION_ID

if START_SNAPSHOT["state"] != "AWAITING_OWNER":
    raise RuntimeError(
        "This live run did not enter AWAITING_OWNER. Inspect the summary above; "
        "model output is nondeterministic, and the walkthrough must not pretend it paused."
    )
print(f"\nDurable run directory: {RUN_DIR}")

$ uv run idg start
{
  "run_id": "28837068bc2b4cbf964e546db869bea3",
  "state": "AWAITING_OWNER",
  "model_invocations": [
    {
      "role": "CODING_AGENT",
      "attempt_number": 1,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    },
    {
      "role": "EVIDENCE_REVIEWER",
      "attempt_number": null,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    }
  ],
  "observed_option": "PRESERVE_EXISTING",
  "classification": "NOT_EVIDENCED",
  "decision_request": {
    "id": "existing_item_sharing_link_rollout",
    "question": "What should happen to existing item-sharing links?",
    "reason": "The gate could not establish from the brief whether the 30-day expiration should apply to existing item-sharing links.",
    "observed": {
      "option": "PRESERVE_EXISTING",
      "behavior": "Existing item-sharing links remain non-expiring with NULL; new lin

## 6. Verify the pinned run envelope

The run stores the commit and the authoritative brief verbatim. It also records the requested model, reasoning effort, invocation role, attempt number, and installed Codex CLI version before each model call. These checks establish the provenance of the pinned inputs and the two model processes used during `start`.

In [6]:
pinned_commit = str(START_SNAPSHOT["base_commit"])
pinned_brief = run_command(
    ["git", "show", f"{pinned_commit}:{BRIEF_PATH}"], show_output=False
).stdout
pinned_schema = run_command(
    ["git", "show", f"{pinned_commit}:{SCHEMA_PATH}"], show_output=False
).stdout

envelope = {
    "run_id": START_SNAPSHOT["run_id"],
    "state": START_SNAPSHOT["state"],
    "base_commit": pinned_commit,
    "created_at": START_SNAPSHOT["created_at"],
    "updated_at": START_SNAPSHOT["updated_at"],
    "original_brief": START_SNAPSHOT["original_brief"],
    "model_invocations": START_SNAPSHOT["model_invocations"],
    "brief_matches_pinned_commit": START_SNAPSHOT["original_brief"] == pinned_brief,
    "schema_matches_pre_start_commit": pinned_schema == schema_at_head,
}
print(json.dumps(envelope, indent=2))
attempt_one_prompt = str(START_SNAPSHOT["attempts"][0]["coding_prompt"])
assert pinned_commit == HEAD_BEFORE_START
assert pinned_brief in attempt_one_prompt
assert pinned_schema in attempt_one_prompt
assert [record["role"] for record in envelope["model_invocations"]] == [
    "CODING_AGENT",
    "EVIDENCE_REVIEWER",
]
assert all(record["model"] == CODEX_MODEL for record in envelope["model_invocations"])
assert all(
    record["reasoning_effort"] == CODEX_REASONING_EFFORT for record in envelope["model_invocations"]
)
assert envelope["brief_matches_pinned_commit"]
assert envelope["schema_matches_pre_start_commit"]

{
  "run_id": "28837068bc2b4cbf964e546db869bea3",
  "state": "AWAITING_OWNER",
  "base_commit": "43c97f5306c14d26043b8dfb25dc79f866167876",
  "created_at": "2026-08-23T18:12:29.438439Z",
  "updated_at": "2026-08-23T18:12:43.251102Z",
  "original_brief": "Add 30-day expiration support to item-sharing links.\n\nStore expiration in `public.share_links.expires_at` as a nullable timestamp with time\nzone. New item-sharing links must expire 30 days after creation.\n",
  "model_invocations": [
    {
      "role": "CODING_AGENT",
      "attempt_number": 1,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    },
    {
      "role": "EVIDENCE_REVIEWER",
      "attempt_number": null,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    }
  ],
  "brief_matches_pinned_commit": true,
  "schema_matches_pre_start_commit": true
}


## 7. Inspect the first coding request

This is the exact project-controlled prompt persisted for attempt one. It is the materialized execution envelope described above: isolation instructions followed by the original brief and baseline schema. The product requirements occur only inside the verbatim brief. The coding agent receives no typed answer for the omitted existing-link policy.

In [7]:
from implicit_decision_gate.codex_client import CODING_SCHEMA

attempt_one = START_SNAPSHOT["attempts"][0]
print("Persisted coding prompt:\n")
print(attempt_one["coding_prompt"])
print("\nCodex structured-output schema:\n")
print(json.dumps(CODING_SCHEMA, indent=2))

Persisted coding prompt:

You create exactly one PostgreSQL migration.
Use only the supplied brief and baseline schema. Do not inspect or edit repository files.
Return the complete migration as structured SQL, without transaction-control statements.

Original brief:
Add 30-day expiration support to item-sharing links.

Store expiration in `public.share_links.expires_at` as a nullable timestamp with time
zone. New item-sharing links must expire 30 days after creation.


Baseline schema:
CREATE TABLE public.share_links (
    id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    token text NOT NULL UNIQUE,
    created_at timestamp with time zone NOT NULL DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO public.share_links (token) VALUES ('existing-fixture');


Codex structured-output schema:

{
  "type": "object",
  "properties": {
    "sql": {
      "type": "string"
    }
  },
  "required": [
    "sql"
  ],
  "additionalProperties": false
}


## 8. Inspect the first proposed migration

The migration has three useful representations in this system: SQL as the proposed mechanism, a hash-identified immutable artifact, and normalized behavior produced by PostgreSQL. This cell shows the first two and independently recomputes the artifact digest.

In [8]:
artifact_one_path = RUN_DIR / "attempt-1.sql"
migration_one = artifact_one_path.read_text(encoding="utf-8")
computed_digest_one = hashlib.sha256(migration_one.encode()).hexdigest()
worktree_one = Path(str(attempt_one["worktree_path"]))
worktree_one_head = run_command(
    ["git", "-C", str(worktree_one), "rev-parse", "HEAD"],
    show_output=False,
).stdout.strip()
worktree_one_status = run_command(
    ["git", "-C", str(worktree_one), "status", "--short"],
    show_output=False,
).stdout.rstrip()

print("Attempt-one SQL artifact:\n")
print(migration_one)
artifact_one = {
    "path": str(artifact_one_path),
    "stored_digest": attempt_one["artifact_digest"],
    "computed_digest": computed_digest_one,
    "digest_matches": attempt_one["artifact_digest"] == computed_digest_one,
    "file_mode": oct(stat.S_IMODE(artifact_one_path.stat().st_mode)),
    "worktree_path": str(worktree_one),
    "clean_start_verified_before_write": attempt_one["clean_start_verified"],
    "worktree_head": worktree_one_head,
    "worktree_matches_base_commit": worktree_one_head == pinned_commit,
    "worktree_status_after_write": worktree_one_status,
}
print(json.dumps(artifact_one, indent=2))
assert artifact_one["digest_matches"]
assert artifact_one["worktree_matches_base_commit"]

Attempt-one SQL artifact:

ALTER TABLE public.share_links
  ADD COLUMN expires_at timestamp with time zone;

ALTER TABLE public.share_links
  ALTER COLUMN expires_at SET DEFAULT (CURRENT_TIMESTAMP + INTERVAL '30 days');

{
  "path": "/home/user/projects/implicit-decision-gate/.idg/runs/28837068bc2b4cbf964e546db869bea3/attempt-1.sql",
  "stored_digest": "13054ad10860ced9ae1dd72bd7799d234a8f3ccc692d3c43d69a9ab7820260e9",
  "computed_digest": "13054ad10860ced9ae1dd72bd7799d234a8f3ccc692d3c43d69a9ab7820260e9",
  "digest_matches": true,
  "file_mode": "0o444",
  "worktree_path": "/home/user/projects/.implicit-decision-gate-idg-worktrees/28837068bc2b4cbf964e546db869bea3/attempt-1",
  "clean_start_verified_before_write": true,
  "worktree_head": "43c97f5306c14d26043b8dfb25dc79f866167876",
  "worktree_matches_base_commit": true,
  "worktree_status_after_write": "?? examples/share-link-expiration/migrations/idg-28837068bc2b4cbf964e546db869bea3-attempt-1.sql"
}


## 9. Inspect normalized runtime evidence

The verifier applied the baseline schema and migration in a disposable database, inspected the column and two representative rows, then rolled the transaction back. A modeled policy requires the exact PostgreSQL type, a nullable column, a non-null default, and one immediate insert whose expiration is within 10 seconds of migration time plus 30 days. The existing-row label then distinguishes the two accepted policies. Anything else is `UNMODELED` and fails before evidence review.

The raw timestamps exist only during the probe. `run.json` retains the normalized facts below. This bounded check doesn't prove how an arbitrary future link relates to its own `created_at`. The `ObservationResult` is the migration's behavioral representation for this demo contract.

In [9]:
observation_one = attempt_one["observation"]
print(json.dumps(observation_one, indent=2))

{
  "data_type": "timestamp with time zone",
  "nullable": true,
  "column_default": "(CURRENT_TIMESTAMP + '30 days'::interval)",
  "insert_without_value": "approximately_now_plus_30_days",
  "existing_row": "null",
  "rollout_option": "PRESERVE_EXISTING",
  "rollback_verified": true
}


## 10. Inspect the narrow evidence review

The reviewer does not analyze every meaning in the brief. For each decision reported by the observer, it receives the exact brief plus one observed hypothesis and asks whether that behavior is explicitly supported. Its structured-output schema requires a `classification` from `SUPPORTED`, `CONTRADICTED`, `NOT_EVIDENCED`, or `UNCERTAIN`, plus an `evidence_quote` string, with no additional fields. The adapter stores an empty quote as `null`.

The persisted AI-derived result is only `{classification, evidence_quote}`. Supported or contradicted quotes are additionally checked as literal substrings of the brief.

In [10]:
print("Persisted reviewer prompt:\n")
print(START_DECISION["reviewer_prompt"])
print("\nValidated reviewer result:\n")
print(json.dumps(START_DECISION["reviewer_result"], indent=2))

# This is an inspection view over persisted fields, not another stored model.
brief_after_review_view = {
    "authoritative_brief": START_SNAPSHOT["original_brief"],
    "observed_hypothesis": START_DECISION["observed"],
    "reviewer_result": START_DECISION["reviewer_result"],
}
print("\nWhat exists after review:\n")
print(json.dumps(brief_after_review_view, indent=2))

Persisted reviewer prompt:

Classify whether the brief explicitly supports the observed existing-row behavior.
Return SUPPORTED, CONTRADICTED, NOT_EVIDENCED, or UNCERTAIN. SUPPORTED and
CONTRADICTED require an exact quote from the brief; otherwise set evidence_quote
to an empty string.

Original brief:
Add 30-day expiration support to item-sharing links.

Store expiration in `public.share_links.expires_at` as a nullable timestamp with time
zone. New item-sharing links must expire 30 days after creation.


Observed rollout option: PRESERVE_EXISTING
Observed behavior: Existing item-sharing links remain non-expiring with NULL; new links default to approximately 30 days after creation; expires_at remains nullable.

Validated reviewer result:

{
  "classification": "NOT_EVIDENCED",
  "evidence_quote": null
}

What exists after review:

{
  "authoritative_brief": "Add 30-day expiration support to item-sharing links.\n\nStore expiration in `public.share_links.expires_at` as a nullable timesta

### Gate logic

The reviewer doesn't decide the rollout policy. Its classification determines whether the observed behavior is already justified, conflicts with the brief, or exposes missing intent. Only the missing-intent branch asks the owner.

<img src="assets/diagrams/gate_logic.png" alt="Gate logic: modeled behavior is reviewed against the brief; missing evidence pauses for the owner, and the resumed attempt completes only when its observed behavior matches the selected option." width="480">

[Review the Mermaid source.](assets/diagrams/gate_logic.mmd)

## 11. Inspect the durable pause and typed question

Any `NOT_EVIDENCED` or `UNCERTAIN` review maps the run to `AWAITING_OWNER`. The fixed application vocabulary turns each unsupported observed policy into a narrow request with verifiable choices. `decision_requests` is derived for presentation, while `run.json` persists the ordered `decisions` records.

In [11]:
pause_summary = run_idg("show", RUN_ID)
print("\nPersisted decision record:\n")
print(json.dumps(START_DECISION, indent=2))
assert pause_summary["state"] == "AWAITING_OWNER"
assert len(pause_summary["decision_requests"]) == 1

$ uv run idg show 28837068bc2b4cbf964e546db869bea3
{
  "run_id": "28837068bc2b4cbf964e546db869bea3",
  "state": "AWAITING_OWNER",
  "model_invocations": [
    {
      "role": "CODING_AGENT",
      "attempt_number": 1,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    },
    {
      "role": "EVIDENCE_REVIEWER",
      "attempt_number": null,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    }
  ],
  "observed_option": "PRESERVE_EXISTING",
  "classification": "NOT_EVIDENCED",
  "decision_request": {
    "id": "existing_item_sharing_link_rollout",
    "question": "What should happen to existing item-sharing links?",
    "reason": "The gate could not establish from the brief whether the 30-day expiration should apply to existing item-sharing links.",
    "observed": {
      "option": "PRESERVE_EXISTING",
      "behavior": "Existing item-sharing links remain

## 12. Complete the missing contract

Inspect the observed option and the two behaviors above, then review or edit the literal below. Either choice is valid: selecting the observed option explicitly confirms it, while selecting the other option makes a behavioral change easier to see in the SQL diff. The checked-in run records `EXPIRE_EXISTING`. The value is an explicit owner input and is not derived by a model or orchestration rule.

In [12]:
OBSERVED_OPTION = str(START_DECISION["observed"])

# Human decision: edit this one value after reading the decision request.
OWNER_OPTION = "EXPIRE_EXISTING"

valid_owner_options = {"PRESERVE_EXISTING", "EXPIRE_EXISTING"}
if OWNER_OPTION not in valid_owner_options:
    raise ValueError(f"OWNER_OPTION must be one of {sorted(valid_owner_options)}")
policy_changed = OWNER_OPTION != OBSERVED_OPTION
print(f"Observed: {OBSERVED_OPTION}")
print(f"Owner selected: {OWNER_OPTION}")
print(f"Behavioral policy changed: {policy_changed}")
if not policy_changed:
    print("The owner confirmed the observed policy; this is a valid contract completion.")

Observed: PRESERVE_EXISTING
Owner selected: EXPIRE_EXISTING
Behavioral policy changed: True


`answer` accepts one typed option for one decision. It doesn't call a model or interpret free text. A run advances to `READY_TO_RESUME` only after every requested decision has an answer. This default scenario has one request. The workspace export scenario demonstrates two answers collected before one retry. This prototype trusts the local caller and doesn't authenticate or attribute the owner identity.

In [13]:
answer_summary = run_idg("answer", RUN_ID, "--decision", DECISION_ID, "--option", OWNER_OPTION)
ANSWER_SNAPSHOT = copy.deepcopy(load_run(RUN_ID))
ANSWER_DECISION = ANSWER_SNAPSHOT["decisions"][0]
print("\nPersisted owner decision:\n")
print(json.dumps(ANSWER_DECISION, indent=2))
assert answer_summary["state"] == "READY_TO_RESUME"

$ uv run idg answer 28837068bc2b4cbf964e546db869bea3 --option EXPIRE_EXISTING
{
  "run_id": "28837068bc2b4cbf964e546db869bea3",
  "state": "READY_TO_RESUME",
  "model_invocations": [
    {
      "role": "CODING_AGENT",
      "attempt_number": 1,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    },
    {
      "role": "EVIDENCE_REVIEWER",
      "attempt_number": null,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    }
  ],
  "observed_option": "PRESERVE_EXISTING",
  "classification": "NOT_EVIDENCED",
  "decision_request": null,
  "owner_option": "EXPIRE_EXISTING",
  "attempt_digests": [
    "13054ad10860ced9ae1dd72bd7799d234a8f3ccc692d3c43d69a9ab7820260e9"
  ],
  "final_worktree_path": null,
  "error": null
}

Persisted owner decision:

{
  "decision_id": "existing_item_sharing_link_rollout",
  "observed": "PRESERVE_EXISTING",
  "selected": "EXPIRE_EXI

## 13. Resume from durable state

`resume` can run later or in another process. It loads the recorded answer, creates a second clean worktree at the original commit, starts a fresh ephemeral Codex process, and probes the regenerated migration. The first model process is not resumed.

In [14]:
resume_summary = run_idg("resume", RUN_ID)
FINAL_SNAPSHOT = copy.deepcopy(load_run(RUN_ID))
attempt_two = FINAL_SNAPSHOT["attempts"][1]
assert len(FINAL_SNAPSHOT["attempts"]) == 2
assert [record["role"] for record in FINAL_SNAPSHOT["model_invocations"]] == [
    "CODING_AGENT",
    "EVIDENCE_REVIEWER",
    "CODING_AGENT",
]
assert FINAL_SNAPSHOT["model_invocations"][-1]["attempt_number"] == 2
assert all(record["model"] == CODEX_MODEL for record in FINAL_SNAPSHOT["model_invocations"])
assert all(
    record["reasoning_effort"] == CODEX_REASONING_EFFORT
    for record in FINAL_SNAPSHOT["model_invocations"]
)

$ uv run idg resume 28837068bc2b4cbf964e546db869bea3
{
  "run_id": "28837068bc2b4cbf964e546db869bea3",
  "state": "COMPLETED",
  "model_invocations": [
    {
      "role": "CODING_AGENT",
      "attempt_number": 1,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    },
    {
      "role": "EVIDENCE_REVIEWER",
      "attempt_number": null,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    },
    {
      "role": "CODING_AGENT",
      "attempt_number": 2,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    }
  ],
  "observed_option": "EXPIRE_EXISTING",
  "classification": "NOT_EVIDENCED",
  "decision_request": null,
  "owner_option": "EXPIRE_EXISTING",
  "attempt_digests": [
    "13054ad10860ced9ae1dd72bd7799d234a8f3ccc692d3c43d69a9ab7820260e9",
    "f0c9612db228aa277a7e73c0efc212b404c18b65ca

## 14. Inspect the fresh second request

Attempt two receives the original brief and schema plus the authoritative owner choice, its required behavior, and PostgreSQL-specific acceptance criteria. It doesn't receive attempt one's SQL or the reviewer's rationale.

In [15]:
prompt_two = str(attempt_two["coding_prompt"])
print("Persisted attempt-two coding prompt:\n")
print(prompt_two)

isolation_checks = {
    "different_worktree": attempt_two["worktree_path"] != attempt_one["worktree_path"],
    "second_clean_start_verified": attempt_two["clean_start_verified"],
    "attempt_one_sql_absent": migration_one.strip() not in prompt_two,
    "reviewer_prompt_absent": START_DECISION["reviewer_prompt"] not in prompt_two,
    "owner_decision_present": (
        f"Authoritative owner decision for {DECISION_ID}: {OWNER_OPTION}" in prompt_two
    ),
}
print("\nContext isolation checks:\n")
print(json.dumps(isolation_checks, indent=2))
assert all(isolation_checks.values())

Persisted attempt-two coding prompt:

You create exactly one PostgreSQL migration.
Use only the supplied brief and baseline schema. Do not inspect or edit repository files.
Return the complete migration as structured SQL, without transaction-control statements.

Original brief:
Add 30-day expiration support to item-sharing links.

Store expiration in `public.share_links.expires_at` as a nullable timestamp with time
zone. New item-sharing links must expire 30 days after creation.


Baseline schema:
CREATE TABLE public.share_links (
    id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    token text NOT NULL UNIQUE,
    created_at timestamp with time zone NOT NULL DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO public.share_links (token) VALUES ('existing-fixture');


Authoritative owner decision: EXPIRE_EXISTING
Required behavior: Existing item-sharing links receive an expiration approximately 30 days from migration; new links default to approximately 30 days after creation; expires_at rem

## 15. Compare the two proposed mechanisms

The unified diff makes the agent's implementation change inspectable. It's not a proof of correctness which comes from the second PostgreSQL probe in the next stage. The metadata also demonstrates separate worktrees and immutable digests.

In [16]:
artifact_two_path = RUN_DIR / "attempt-2.sql"
migration_two = artifact_two_path.read_text(encoding="utf-8")
computed_digest_two = hashlib.sha256(migration_two.encode()).hexdigest()
migration_diff = "".join(
    difflib.unified_diff(
        migration_one.splitlines(keepends=True),
        migration_two.splitlines(keepends=True),
        fromfile="attempt-1.sql",
        tofile="attempt-2.sql",
    )
)

print("Attempt-two SQL artifact:\n")
print(migration_two)
print("Unified diff:\n")
print(migration_diff or "No textual difference between the two migrations.")
comparison = {
    "attempt_1": {
        "digest": attempt_one["artifact_digest"],
        "worktree": attempt_one["worktree_path"],
    },
    "attempt_2": {
        "digest": attempt_two["artifact_digest"],
        "computed_digest": computed_digest_two,
        "digest_matches": attempt_two["artifact_digest"] == computed_digest_two,
        "worktree": attempt_two["worktree_path"],
    },
}
print(json.dumps(comparison, indent=2))
assert comparison["attempt_2"]["digest_matches"]

Attempt-two SQL artifact:

ALTER TABLE public.share_links
  ADD COLUMN expires_at timestamp with time zone;

UPDATE public.share_links
SET expires_at = CURRENT_TIMESTAMP + INTERVAL '30 days'
WHERE expires_at IS NULL;

ALTER TABLE public.share_links
  ALTER COLUMN expires_at SET DEFAULT (CURRENT_TIMESTAMP + INTERVAL '30 days');

Unified diff:

--- attempt-1.sql
+++ attempt-2.sql
@@ -1,5 +1,9 @@
 ALTER TABLE public.share_links
   ADD COLUMN expires_at timestamp with time zone;
 
+UPDATE public.share_links
+SET expires_at = CURRENT_TIMESTAMP + INTERVAL '30 days'
+WHERE expires_at IS NULL;
+
 ALTER TABLE public.share_links
   ALTER COLUMN expires_at SET DEFAULT (CURRENT_TIMESTAMP + INTERVAL '30 days');

{
  "attempt_1": {
    "digest": "13054ad10860ced9ae1dd72bd7799d234a8f3ccc692d3c43d69a9ab7820260e9",
    "worktree": "/home/user/projects/.implicit-decision-gate-idg-worktrees/28837068bc2b4cbf964e546db869bea3/attempt-1"
  },
  "attempt_2": {
    "digest": "f0c9612db228aa277a7e73c0efc212b404

## 16. Verify the completed contract

PostgreSQL again reduces runtime behavior to the bounded `ObservationResult`. Final acceptance is the typed equality shown below: the observed rollout must equal the owner's selected rollout. No model judges whether attempt two succeeded.

In [17]:
observation_two = attempt_two["observation"]
final_decision = FINAL_SNAPSHOT["decisions"][0]
selected_option = str(final_decision["selected"])
observed_option_two = str(observation_two["outcomes"][DECISION_ID])
selected_matches_observed = selected_option == observed_option_two
final_verification = {
    "selected_by_owner": selected_option,
    "observed_by_postgresql": observed_option_two,
    "selected_equals_observed": selected_matches_observed,
    "final_state": FINAL_SNAPSHOT["state"],
    "error": FINAL_SNAPSHOT["error"],
    "observation": observation_two,
}
print(json.dumps(final_verification, indent=2))
assert selected_matches_observed
assert resume_summary["state"] == "COMPLETED"

{
  "selected_by_owner": "EXPIRE_EXISTING",
  "observed_by_postgresql": "EXPIRE_EXISTING",
  "selected_equals_observed": true,
  "final_state": "COMPLETED",
  "error": null,
  "probe_result": {
    "data_type": "timestamp with time zone",
    "nullable": true,
    "column_default": "(CURRENT_TIMESTAMP + '30 days'::interval)",
    "insert_without_value": "approximately_now_plus_30_days",
    "existing_row": "approximately_migration_time_plus_30_days",
    "rollout_option": "EXPIRE_EXISTING",
    "rollback_verified": true
  }
}


## 17. Inspect the durable record

The complete `run.json` below is the atomically replaced current-state snapshot. Its ordered model invocations, attempts, decision timestamps, prompts, observations, and adjacent immutable SQL files are the inspectable state of this run.

In [18]:
run_json_path = RUN_DIR / "run.json"
print("Complete run.json:\n")
print(run_json_path.read_text(encoding="utf-8"))

timeline = [
    {"event": "run_created", "at": FINAL_SNAPSHOT["created_at"]},
    {
        "event": "attempt_1_completed",
        "at": FINAL_SNAPSHOT["attempts"][0]["completed_at"],
    },
    {"event": "owner_answered", "at": final_decision["answered_at"]},
    {
        "event": "attempt_2_completed",
        "at": FINAL_SNAPSHOT["attempts"][1]["completed_at"],
    },
    {"event": "snapshot_updated", "at": FINAL_SNAPSHOT["updated_at"]},
]
artifacts = [
    {"name": path.name, "bytes": path.stat().st_size}
    for path in sorted(RUN_DIR.glob("attempt-*.sql"))
]
print("Derived timeline:\n")
print(json.dumps(timeline, indent=2))
print("\nImmutable SQL artifacts:\n")
print(json.dumps(artifacts, indent=2))

Complete run.json:

{
  "run_id": "28837068bc2b4cbf964e546db869bea3",
  "state": "COMPLETED",
  "original_brief": "Add 30-day expiration support to item-sharing links.\n\nStore expiration in `public.share_links.expires_at` as a nullable timestamp with time\nzone. New item-sharing links must expire 30 days after creation.\n",
  "base_commit": "43c97f5306c14d26043b8dfb25dc79f866167876",
  "model_invocations": [
    {
      "role": "CODING_AGENT",
      "attempt_number": 1,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    },
    {
      "role": "EVIDENCE_REVIEWER",
      "attempt_number": null,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    },
    {
      "role": "CODING_AGENT",
      "attempt_number": 2,
      "model": "gpt-5.6-terra",
      "reasoning_effort": "xhigh",
      "codex_cli_version": "codex-cli 0.149.0"
    }
  ],
  "attempts": [
    {
 

## 18. Current boundary

What exists in this implementation:

- Commit-pinned authoritative briefs and technical contexts.
- Application-pinned `gpt-5.6-terra` model calls at `xhigh` with per-invocation provenance.
- A coding agent constrained to one structured scenario artifact.
- A PostgreSQL observer for migration behavior and a network-disabled Python observer for workspace export authorization.
- One narrow AI evidence review per observed hypothesis.
- One durable pause that can collect multiple typed human amendments before one clean-context retry.
- A deterministic final comparison between every expected and observed behavior.

What doesn't exist yet:

- A general semantic compiler that turns arbitrary prose into a trusted typed contract.
- Authenticated and attributable agent or human identity.
- Persisted raw observer output or raw model transcripts.
- An append-only event ledger or graph database.

A future semantic verifier could let AI propose typed claims, unknown fields, and source spans for human approval, then compare the approved policy with normalized runtime evidence. The AI-derived interpretation shouldn't replace the human-owned brief. The current reviewer is also deliberately weaker than that vision: literal quote validation only proves source occurrence, but not semantic relevance.

## 19. Remove the disposable verifier

Stopping Compose removes the local verifier container and discards its temporary database state after the walkthrough. It doesn't remove `.idg/runs/<run_id>`, the SQL artifacts, or the detached worktrees, so the durable evidence remains available for manual inspection.

In [19]:
_ = run_command(["docker", "compose", "down", "--volumes"])

$ docker compose down --volumes
 Container implicit-decision-gate-postgres-1 Stopping 
 Container implicit-decision-gate-postgres-1 Stopped 
 Container implicit-decision-gate-postgres-1 Removing 
 Container implicit-decision-gate-postgres-1 Removed 
 Network implicit-decision-gate_default Removing 
 Network implicit-decision-gate_default Removed


## 20. How the pattern scales

The reusable idea is a contract-completion interface:

```text
authoritative contract
→ agent action
→ trusted runtime observation
→ normalized typed outcomes
→ deterministic comparison
→ owner decisions for any unknowns
```

For PostgreSQL, additional evidence adapters could verify backfills, nullability changes, uniqueness and foreign-key constraints, deletion behavior, indexes, transactional safety, rollback behavior, and compatibility with representative existing rows.

The implemented workspace export observer applies the same lifecycle to two choices at once. A disposable, network-disabled Python container calls the generated handler twice as an owner with shared state and once each as an administrator and member, then records each status and job count.

The same lifecycle can extend to other surfaces:

| Surface | Trusted observation | Example typed facts |
| --- | --- | --- |
| Events and queues | Test broker, schema registry, and recorded consumer behavior | Schema version, routing, ordering, delivery, and idempotency |
| Infrastructure | Plan output plus cloud control-plane observations | Resource changes, network exposure, encryption, and retention |
| Builds and deployments | Signed artifacts, test results, and rollout telemetry | Artifact digest, required checks, health, and rollback outcome |

The durable state machine, evidence provenance, typed owner decisions, clean retry, and deterministic comparison can be reused across these use cases. Each use case still needs its own trusted observer, normalization adapter, and bounded vocabulary. AI can help extract proposed claims or classify source evidence, but it shouldn't be the authority for either the contract or the observed truth.

In a 1Password integration, Verified Loops would continue to provide authenticated human and agent identity, controlled tool access, attributable evidence, and permission enforcement. This stage would only complete missing intent and return the amended contract and verified result to that wider architecture.